In [1]:
import sys
from pprint import pprint
import os
from dotenv import load_dotenv

from ckan_batch.client import CKANClient
from ckan_batch.reader.pidinst import read_pidinst_template

In [2]:
# temp_path = r"C:\Users\mot032\CSIRO\AuScope AVRE - Persistent ID for Instruments (PIDINST)\templates\PIDINST_Batch_Instruments_Template_SingleSheet.xlsx"
# temp_path = r"C:\Users\mot032\CSIRO\AuScope AVRE - Persistent ID for Instruments (PIDINST)\data\PIDINST_Batch_BenKay.xlsx"
temp_path = r"C:\Drive_D\projects\avre\apps\PIDINST\AuScope-instrument-registry\ckan_batch\src\ckan_batch\reader\templates\PIDINST.xlsx"

load_dotenv()

CKAN_BASE_URL = os.getenv("CKAN_BASE_URL", "").rstrip("/")
CKAN_API_KEY = os.getenv("CKAN_API_KEY")  # required for Create/Update/Delete (CUD) actions

# INITIALISE CLIENT

In [3]:
ckan_client = CKANClient(CKAN_BASE_URL, apikey=CKAN_API_KEY)

# Template Reader

In [7]:
res_map = read_pidinst_template(temp_path, client=ckan_client)
pkg_dict = res_map.records
if len(res_map.errors) > 0:
    print("=============ERRORS==========\n")
    pprint(res_map.errors)

print(len(pkg_dict))
# pprint(pkg_dict[0])

1


In [8]:
pkg_dict

[{'model': [{'model_name': 'CGS-T',
    'model_identifier': None,
    'model_identifier_type': None}],
  'owner': [{'owner_party_id': 'curtin-hub-for-immersive-visualisation-and-eresearch',
    'owner_name': 'Curtin Hub for Immersive Visualisation and eResearch',
    'owner_identifier': 'https://ror.org/00hjft591',
    'owner_identifier_type': 'ROR',
    'owner_contact': 'contact@example.org'}],
  'date': [{'date_value': '2026-01-15', 'date_type': 'Commissioned'}],
  'measured_variable': [{'measured_variable_name': 'temperature',
    'measured_variable_identifier': 'https://gcmd.earthdata.nasa.gov/kms/concept/5819b364-892c-40ab-a90f-5a239c3e028b',
    'measured_variable_identifier_type': 'URL'},
   {'measured_variable_name': 'pressure',
    'measured_variable_identifier': 'https://gcmd.earthdata.nasa.gov/kms/concept/6f900066-a11f-43d3-9ef9-8d2b153c788e',
    'measured_variable_identifier_type': 'URL'},
   {'measured_variable_name': 'Magnetic Field',
    'measured_variable_identifier': 

# Get all packages

In [14]:
# ckan_client.get_all(verbose=True)[0]

# Create Instruments

In [12]:
res = ckan_client.create_records(
    records=pkg_dict,
    make_public=False,
    dry_run = False,
    record_type="instrument",
    allow_update_if_exists = False,
)

In [13]:
res.failed

[{'index': 1,
  'title': 'Example Instrument',
  'error': 'CKANAPIError',
  'ckan_error': 'Solr returned an error: Solr responded with an error (HTTP 400): [Reason: ERROR: [doc=634846bd0e69a837832dfae57dfa2625] multiple values encountered for non multiValued field alternate_identifier: [SN-000099, ALT-123]]',
  'payload': {'model': '[{"model_name": "CGS-T", "model_identifier": null, "model_identifier_type": null}]',
   'owner': '[{"owner_party_id": "curtin-hub-for-immersive-visualisation-and-eresearch", "owner_name": "Curtin Hub for Immersive Visualisation and eResearch", "owner_identifier": "https://ror.org/00hjft591", "owner_identifier_type": "ROR", "owner_contact": "contact@example.org"}]',
   'date': '[{"date_value": "2026-01-15", "date_type": "Commissioned"}]',
   'measured_variable': '[{"measured_variable_name": "temperature", "measured_variable_identifier": "https://gcmd.earthdata.nasa.gov/kms/concept/5819b364-892c-40ab-a90f-5a239c3e028b", "measured_variable_identifier_type": "U

# Delete

In [6]:
pkgs = ckan_client.get_records_by_title(
    "Phoenix Geophysics MTU-5C Magnetotelluric Data Acquisition System"
)
for pkg in pkgs:
    print(pkg.get("title"))

In [5]:
for pkg in pkgs:
    pkg_id = pkg.get("id")
    ckan_client.delete_record_by_id(record_id=pkg_id, hard_delete=True)

CKANAPIError deleting record 38a7d576-dcf1-4c29-a760-28aac2183ddc: ['https://instrument-test.data.auscope.org.au/api/action/package_purge', 400, '"Bad request - Action name not known: package_purge"']
CKANAPIError deleting record 26dc281a-cfd6-4b7d-87c1-4f4dbb3f2c0f: ['https://instrument-test.data.auscope.org.au/api/action/package_purge', 400, '"Bad request - Action name not known: package_purge"']
CKANAPIError deleting record 66582c88-35c5-4b55-92fd-2cfc33f93bd3: ['https://instrument-test.data.auscope.org.au/api/action/package_purge', 400, '"Bad request - Action name not known: package_purge"']


In [8]:
to_delete = ckan_client.delete_all_in_org(
    owner_org="auscope-org",
    dry_run=False,
    dataset_type='instrument',
    include_draft = True,
    include_private = True,
    include_public = True,
    hard_delete = True
)

Found 0 instrument dataset(s) in owner_org='auscope-org' for HARD DELETE (draft=True, private=True, public=True)

Deleted: 0


# Parties

In [4]:
ckan_client.get_all_parties(verbose=True)

[{'aliases': '',
  'approval_status': 'approved',
  'created': '2026-03-16T09:34:59.220038',
  'description': '',
  'display_name': 'AuScope',
  'id': 'c410b75a-d9e9-476f-ae7d-cc2652021440',
  'image_display_url': '',
  'image_url': '',
  'is_organization': False,
  'name': 'auscope',
  'num_followers': 0,
  'package_count': 2,
  'parent_party': '',
  'party_contact': '',
  'party_identifier': '',
  'party_identifier_ror': 'https://ror.org/04s1m4564',
  'party_identifier_type': 'ROR',
  'party_role': ['Owner', 'Funder'],
  'party_state': 'Victoria',
  'ror_country': 'Australia',
  'ror_hierarchy_display': 'AuScope',
  'ror_parents_json': '[]',
  'ror_types': 'facility',
  'short_description': '',
  'state': 'active',
  'title': 'AuScope',
  'type': 'party',
  'website': 'https://www.auscope.org.au/'},
 {'aliases': '',
  'approval_status': 'approved',
  'created': '2026-03-16T09:34:38.635868',
  'description': '',
  'display_name': 'Curtin Hub for Immersive Visualisation and eResearch',

In [25]:
ckan_client.delete_all_parties(
    dry_run=False,
    hard_delete=True
)

Found 1 party group(s) for HARD DELETE
 - curtin-hub-for-immersive-visualisation-and-eresearch (8f8a16a2-70df-4cdc-91ce-f8192f320930) | Curtin Hub for Immersive Visualisation and eResearch | state=active

Deleted: 1


[{'id': '8f8a16a2-70df-4cdc-91ce-f8192f320930',
  'name': 'curtin-hub-for-immersive-visualisation-and-eresearch',
  'title': 'Curtin Hub for Immersive Visualisation and eResearch',
  'type': 'party',
  'state': 'active'}]

In [6]:
ckan_client.action.taxonomy_list()

[{'id': '9d93b7ac-53e5-432b-a607-408f83902525',
  'name': 'instruments',
  'title': 'Instruments',
  'uri': 'https://instrument-test.data.auscope.org.au/taxonomies/Instruments'},
 {'id': '2c1ae2d4-8c1d-4490-872a-a6527eba59c1',
  'name': 'measured-variables',
  'title': 'Measured Variables',
  'uri': 'https://instrument-test.data.auscope.org.au/taxonomies/Measured Variables'},
 {'id': '0baab70f-bd96-4632-b802-ce0dae4d748a',
  'name': 'platforms',
  'title': 'Platforms',
  'uri': 'https://instrument-test.data.auscope.org.au/taxonomies/Platforms'}]

In [7]:
ckan_client.action.taxonomy_term_list(id='9d93b7ac-53e5-432b-a607-408f83902525')

[{'id': '15ba49f1-c839-4998-904d-d1abd4bb5c8b',
  'label': 'Magento-test',
  'description': '',
  'uri': 'https://instrument-test.data.auscope.org.au/taxonomies/term/15ba49f1-c839-4998-904d-d1abd4bb5c8b',
  'extras': {},
  'taxonomy_id': '9d93b7ac-53e5-432b-a607-408f83902525',
  'parent_id': 'ed449903-c98a-44cb-865c-6b9a9fbbc008'},
 {'id': 'ed449903-c98a-44cb-865c-6b9a9fbbc008',
  'label': 'Magnetotelluric data acquisition system',
  'description': '',
  'uri': 'https://instrument-test.data.auscope.org.au/taxonomies/term/ed449903-c98a-44cb-865c-6b9a9fbbc008',
  'extras': {},
  'taxonomy_id': '9d93b7ac-53e5-432b-a607-408f83902525',
  'parent_id': None}]

In [6]:
ckan_client.action.group_list(type="facility")

['adelaide-university',
 'auscope',
 'curtin-hub-for-immersive-visualisation-and-eresearch',
 'curtin-university']

In [7]:
from typing import List, Dict, Any
from ckanapi.errors import CKANAPIError

facility_names = ckan_client.action.group_list(type="facility")

hard_delete = True
dry_run = False  # change to False when ready

to_delete: List[Dict[str, Any]] = [{"name": name} for name in facility_names]

mode = "HARD DELETE" if hard_delete else "SOFT DELETE"
print(f"Found {len(to_delete)} facility group(s) for {mode}")
for g in to_delete[:20]:
    print(f" - {g['name']}")
if len(to_delete) > 20:
    print(f" ... and {len(to_delete) - 20} more")

if dry_run:
    print("\nDRY RUN: no deletions performed.")
else:
    deleted = 0
    failed: List[Dict[str, Any]] = []

    for g in to_delete:
        try:
            if hard_delete:
                try:
                    ckan_client.action.group_delete(id=g["name"])
                except Exception:
                    pass  # maybe already deleted

                ckan_client.action.group_purge(id=g["name"])
            else:
                ckan_client.action.group_delete(id=g["name"])

            deleted += 1

        except CKANAPIError as e:
            failed.append(
                {
                    "group": g,
                    "error": getattr(e, "error_dict", None) or str(e),
                }
            )
        except Exception as e:
            failed.append(
                {
                    "group": g,
                    "error": f"Unexpected error: {e}",
                }
            )

    print(f"\nDeleted: {deleted}")
    if failed:
        print(f"Failed: {len(failed)}")
        for f in failed[:10]:
            print(" -", f)

Found 4 facility group(s) for HARD DELETE
 - adelaide-university
 - auscope
 - curtin-hub-for-immersive-visualisation-and-eresearch
 - curtin-university

Deleted: 4


In [17]:
ckan_client.action.package_search(
    q=f'manufacturer_name:"Phoenix Geophysics"',
                fq="type:instrument",
                include_private=False,
                include_drafts=False,
                rows=100,
)

{'count': 6,
 'facets': {},
 'results': [{'alternate_identifier_obj': '[{"alternate_identifier": "alt 1", "alternate_identifier_name": "", "alternate_identifier_type": "SerialNumber"}]',
   'author': None,
   'author_email': None,
   'citation': 'Phoenix Geophysics test change name (2026): test update inst party. AuScope. https://handle.test.datacite.org/10.83627/tli6d447',
   'creator_user_id': '4041e374-9dba-4357-b6c9-3f5991d0284e',
   'credit': '',
   'date': '',
   'description': '',
   'doi': '10.83627/tli6d447',
   'duplicate_of': '',
   'epsg': '4326 - WGS 84',
   'epsg_code': '4326',
   'funder': '[{"award_number": "", "award_title": "", "award_uri": "", "funder_identifier": "https://ror.org/02n415q13", "funder_identifier_type": "ROR", "funder_name": "Curtin University", "funder_party_id": "curtin-university"}]',
   'id': '6d1129df-161e-4bd3-8bb1-c2e17b3ceefd',
   'instrument_classification': 'Geophysics',
   'instrument_type': '',
   'instrument_type_gcmd': '',
   'instrument_

In [16]:
ckan_client.action.package_search(
    q='*:*',
    fq=f'type:instrument AND manufacturer_name:"Phoenix Geophysics"',
    include_private=False,
    include_drafts=False,
    rows=100,
)

{'count': 0,
 'facets': {},
 'results': [],
 'sort': 'score desc, metadata_modified desc',
 'search_facets': {}}